# 04b — Four views of AI

**Background resource:** Section numbers below refer to the [August perspective](../../paper/background/august-perspective.pdf), preserved alongside the current paper.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/brains-to-models-publication/blob/main/code/notebooks/04b_four_views_of_AI.ipynb)

Companion to §5 of *From Brains to Models*. Russell & Norvig’s 2×2 crosses two distinctions — *human* performance against *rationality*, and *thinking* against *acting*. The paper’s claim is that foundation models sit in every cell at once, and that each cell now returns a different verdict.

This notebook is the map, not a demonstration. No training, no parameters — a catalogue and a drawing, like notebook 00. Notebook 04 is the experiment on the upper-left cell (framing as next-token mass). The longer keynote notebook this is adapted from — ELIZA, a drift–diffusion model, Horn clauses, a vacuum-cleaner agent — lives in the sister repo and is not repeated here.

**Claim.** Foundation models can be scored in every Russell–Norvig cell, and each cell returns a different verdict.

**Method.** A 2×2 catalogue and drawing; no training. Seed 7 for the figure. Notebook 04 is the experiment on the upper-left cell.

**Result.** The map figure; lookup by focus term.

**Limit.** Not an AIMA tutorial and not a second experiment. Verdicts are this paper’s, not Russell and Norvig’s.

<details>
<summary><b>Key terms — for clinicians and neuroscientists</b></summary>

- **Russell & Norvig's 2×2.** A standard map of what "AI" has meant, crossing two questions: is the yardstick *human* performance or ideal *rationality*, and does the system *think* (internal process) or *act* (external behaviour)?
- **Thinking versus acting.** The process/behaviour distinction — modelling how a mind works versus producing the right output whatever the internals.
- **Humanly versus rationally.** Matching what people actually do (errors and all) versus what an ideal reasoner should do.
- **[The imitation game (Turing test)](https://en.wikipedia.org/wiki/Turing_test).** The "acting humanly" cell: indistinguishability from a person in conversation. Foundation models now pass ever more demanding versions.
- **[Foundation model](https://en.wikipedia.org/wiki/Foundation_model) in every cell.** The paper's point — one adapted predictor can be scored as imitation, as psychology, as a scratchpad, and as an agent, and each cell returns a different verdict.

</details>

<details>
<summary><b>Key terms — for the mathematically inclined</b></summary>

- **Thinking rationally.** The "laws of thought": formal logic and probability/decision theory as normative standards for inference.
- **Acting rationally (rational agent).** An agent choosing $a^{\*}=\arg\max_{a}\,\mathbb{E}[\,U\mid a,\text{belief}\,]$ — expected-utility maximisation given its beliefs.
- **Acting humanly.** Operationalised by the [Turing test](https://en.wikipedia.org/wiki/Turing_test): behavioural indistinguishability, silent on internals.
- **Thinking humanly.** Cognitive modelling, tested empirically (notebook 04) by whether the system reproduces human error patterns.
- **One predictor across the grid.** A single $p_\theta(\text{next}\mid\text{past})$, adapted, can be scored in all four cells; the verdicts diverge.
- **Status.** A catalogue and a drawing — no parameters fit; `FOCUS` selects a term. Verdicts are this paper's.

</details>

The same code writes `paper/figures/four_views.pdf` (`make figures`).

Reproducible: `SEED = 7` in the setup cell. Restart and run all to reproduce the figure.

### Concepts used in this notebook

This notebook draws Russell & Norvig's four definitions of AI and places foundation models in each. The two blocks below define the four cells and the distinctions that separate them, first in plain language and then more formally. Both cover the same concepts at a different level of detail.

<details>
<summary><b>Concepts — for clinicians and neuroscientists</b></summary>

- **Acting humanly — the Turing test.** Judged by behaviour: can it converse indistinguishably from a person? Requires language, knowledge, reasoning, and learning.
- **Thinking humanly — cognitive modelling.** Judged by process: does it work the way a human mind works, characteristic mistakes included? (Notebook 04 tests this.)
- **Thinking rationally — the "laws of thought".** Judged against logic: does it reason by valid inference?
- **Acting rationally — the rational agent.** Judged by outcome: does it act to achieve the best expected result given what it knows?
- **Why one system fills all four.** A foundation model can be prompted to imitate, to err like people, to reason step by step, and to act as an agent — so it can be scored in every cell at once.
- **Map, not measurement.** Nothing is trained; the verdict in each cell is this paper's reading, offered for discussion.

</details>

<details>
<summary><b>Concepts — for the mathematically inclined</b></summary>

- **Two axes.** $\{\text{human}, \text{rational}\}\times\{\text{thinking}, \text{acting}\}$, giving four cells.
- **Rational agent.** Chooses $a^{\*}=\arg\max_{a}\,\mathbb{E}[\,U\mid a,\text{belief}\,]$ — expected-utility maximisation.
- **Laws of thought.** Formal logic and probability as normative inference.
- **Turing test.** A behavioural indistinguishability criterion, silent on internal state.
- **Cognitive modelling.** Empirical match to human data, including error patterns.
- **Limit.** A definitional map of *AI*; no cell asks about embodiment or [experience](https://plato.stanford.edu/entries/consciousness/).

</details>

In [1]:
import os
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt

REPO = os.environ.get("BRAINSMODELS_REPO_URL",
                      "https://github.com/arvidl/brains-to-models-publication.git")
COLAB_ROOT = Path("/content/brains-to-models")

if "google.colab" in sys.modules:
    if not (COLAB_ROOT / "code" / "src" / "brainsmodels").is_dir():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(COLAB_ROOT)])
    root = COLAB_ROOT
else:
    here = Path.cwd().resolve()
    root = here.parents[1] if here.name == "notebooks" else here

src = root / "code" / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from brainsmodels.repro import DEFAULT_SEED, set_seed
from brainsmodels.four_views import (
    QUADRANTS,
    format_quadrant,
    lookup,
    render_paper_figure,
    write_figure,
)

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("repo root:", root)
print("seed:", SEED)
print(len(QUADRANTS), "quadrants in the catalogue")

repo root: <checkout>
seed: 7
4 quadrants in the catalogue


## The two axes

**Rows — what we evaluate.** Internal thought processes versus external behaviour.

**Columns — what we compare against.** Human performance (how people actually think and act) versus rationality (doing the right thing given what is known).

The four cells are then: thinking humanly (cognitive science), thinking rationally (laws of thought), acting humanly (Turing’s test), acting rationally (the rational agent). AIMA treated the grid as a way to organise a textbook. The paper treats it as four different answers to “what does a foundation model tell us about being human?”

The third line in each cell of the figure is this paper’s verdict, not Russell & Norvig’s.

In [2]:
for cell in QUADRANTS:
    print(f"{cell.title:<22}  {cell.aima:<20}  {cell.verdict}")

Thinking humanly        Cognitive science     errors follow the training corpus
Thinking rationally     Laws of thought       a scratchpad is not a proof
Acting humanly          Turing test           imitation is no longer informative
Acting rationally       Rational agent        goals are added after training


## Paper figure

Pastel cells after the keynote notebook. The layout is the same 2×2; the extra line is the verdict §5 will argue.

In [3]:
fig = render_paper_figure()
plt.show()
write_figure(root / "paper" / "figures")

wrote <checkout>/paper/figures/four_views.pdf
wrote <checkout>/paper/figures/four_views.png


/var/folders/1g/_4x26lps6mgg1tr73d7g2bmh0000gp/T/ipykernel_49885/1963760631.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('<checkout>/paper/figures/four_views.pdf')

## Look up a cell

Change `FOCUS` to `turing`, `corpus`, `proof`, `goals`, `04`, or `02b`.

In [4]:
FOCUS = "turing"
hits = lookup(FOCUS)
if not hits:
    print("Nothing matched. Try turing, corpus, proof, goals, framing, agent.")
for cell in hits:
    print(format_quadrant(cell))
    print()

Acting humanly  (Turing test)
  axes     behaviour × human
  verdict  imitation is no longer informative
  see      code/notebooks/01_tiny_transformer.ipynb
  Score the public interface of thought: the strings a person might produce. Turing’s game is now too easy. What imitation does not establish is a world-involving belief, a body that can be wrong at a cost, or experience.



## What to notice

**The grid is two questions, not four kinds of machine.** A system can be scored four ways. Foundation models are one architecture scored four ways. That is why they “redistribute the field across all four quadrants at once.”

**The left column carries the brains-to-models comparison.** Thinking humanly and acting humanly take *us* as the standard. The right column takes an ideal. The paper’s later worry — values, experience, a body — lives mostly on the left, because that is where a person could have been.

**The third line is doing work.** “Errors follow the training corpus” is notebook 04. “Imitation is no longer informative” is notebook 01 plus Turing. “A scratchpad is not a proof” is the System-1/2 warning in §5. “Goals are added after training” is notebook 02b. The map is a table of contents for verdicts, not a second experiment.

**This is not the keynote notebook.** That one builds an ELIZA, fits a drift–diffusion model, chains Horn clauses, and scores a vacuum cleaner. Those toys earn their keep in a course. Here the paper already has a transformer, a T-maze, and a framing corpus. Repeating the course would duplicate the demonstration.

## Why this example is here

This notebook is the map for §5, not a tutorial on AIMA. It exists so that the sentence *foundation models sit in every cell, and each cell returns a different verdict* is something you can point at before notebook 04 puts a designed corpus on the table.

The intended readers are the same as the paper’s: clinicians and neuroscientists, and the mathematically inclined who have never opened Russell & Norvig. The two readings below are of the *same* grid.

### For the clinician and the neuroscientist

You already live with a version of this 2×2. A cognitive test scores *thinking humanly* (does the patient perform like other people?). A protocol or a decision rule scores *thinking rationally* (is the inference valid given the evidence?). A Turing-like impression — “they sound like themselves” — scores *acting humanly*. Choosing the treatment that maximises expected benefit given the goals of care scores *acting rationally*.

A language model in the clinic can look like all four at once: it completes a note like a colleague (acting humanly), it can reproduce a framing effect it read about (thinking humanly), it can write a chain of “therefore”s (thinking rationally), and it can propose a plan if you give it a goal (acting rationally). None of those is a body that can be wrong at a cost. The map’s job is only to keep the four scores from collapsing into “the model is intelligent.”

If you change `FOCUS` to `corpus` or `goals`, you get the cell the paper will argue next, in dated form.

### For the mathematically inclined

There is no dynamics here and no loss. A quadrant is a labelled point in $\\{0,1\\}^2$: thought vs behaviour, human vs rational. The only structure is the product of those two bits. The pastel figure is a drawing of that product. The third line in each cell is a *claim* of the paper, not a coordinate.

What the figure is *not*: a partition of architectures, a completeness theorem, or a ranking. Several research programmes sit across cells (cognitive architectures; verification of agents). The claim that survives that mess is directional: once a single predictor is good enough to be scored in every cell, the interesting question is no longer “which definition of AI is this?” but “what does each definition still fail to see?”

If you want an equation, notebook 04 supplies one: $p_\theta(C\mid \text{probe})$ on a corpus that already contains a published bias. This notebook is only why that object is being asked under the heading *thinking humanly*.

### What this has to do with the rest of the paper

Section 2 is acting humanly at toy scale (a transformer that continues text). Section 3.3 is acting rationally at toy scale (preferred observations vs preferred tokens). Section 5’s framing figure is thinking humanly at toy scale. Thinking rationally has no extra toy: the paper’s point is that a longer prefix is still the same objective. Later sections (dynamics, workspace, values) ask what still fails to appear when every cell looks occupied.

Change `FOCUS`. The cells will stay. The claim should too: **one predictor, four scores, four different remainders.**

### Questions for reflection

Not a quiz. Each question can be asked of a brain and of a machine. Write a short answer that a clinician or a neuroscientist, and a mathematically inclined reader, could both test. The two themes are defined in Further reading below.


1. A language model can be scored in every Russell–Norvig cell. Why is that not the same as occupying every cell as a person does? (The [imitation game](https://en.wikipedia.org/wiki/Turing_test) is only one cell.)

2. Thinking humanly versus acting humanly: which score is a cognitive test, which is a public impression, and which does a fluent note confuse?

3. Thinking rationally: a longer “therefore” is still $p(\text{next}\mid\text{past})$. What would a proof add that a prefix cannot?

4. Acting rationally: a plan given a goal is not a goal the system has. What extra structure did notebook 02b put in that this map does not?

5. Thinking rationally has no extra teaching model in the paper. Why is that a finding rather than a missing notebook?

6. Looking up a cell is not a proof that the system meets the definition. Give a clinical example of the same mistake: a score that looks like the construct.

7. What can agreement in one cell establish at a computational level, and what can it not?

8. What does no cell measure? Use [embodiment](https://plato.stanford.edu/entries/embodied-cognition/) or [phenomenal experience](https://plato.stanford.edu/entries/consciousness/) in the answer.

9. If you change `FOCUS`, the cells stay. What claim should stay with them?

10. Name one remainder for each of the four cells in a phrase a mathematician would accept.


### Further reading

Each annotation is written against the two themes: **(i) From brains to models** asks what a comparison can establish at a computational level: the same objective, the same representation, the same dynamical description, or the same functional test, applied on both sides — brain and model (machine). **(ii) What AI reveals about being human** asks what the comparison isolates when the systems (human and machine) diverge: [embodiment](https://plato.stanford.edu/entries/embodied-cognition/), a body that must satisfy [allostasis](https://en.wikipedia.org/wiki/Allostasis) (viability) constraints, [phenomenal experience](https://plato.stanford.edu/entries/consciousness/), and values that originate in need and [homeostasis](https://en.wikipedia.org/wiki/Homeostasis) rather than in training data.

The demonstrations in this notebook are teaching models. They illustrate a principle at a scale that can be run and modified. They are not recordings from cortex, not evaluations of a large language model, and not evidence that a next-token probability is a value.

Bibliographic links open the paper or a stable copy. Linked terms open a short definition.

1. **Russell, S., and Norvig, P. (2021).** [*Artificial Intelligence: A Modern Approach*](https://aima.cs.berkeley.edu/). 4th ed., Pearson. Book site.
   - *(i) From brains to models.* The 2×2 this notebook draws: process vs behaviour; human performance vs an ideal of rationality. Chapter 1 is the source of the cells; the third line in each cell is this paper’s verdict for foundation models, not AIMA’s.
   - *(ii) What AI reveals about being human.* No cell asks whether the system has a body or an experience. The map is complete as a definition of *AI*. It is incomplete as a definition of being human.

2. **Turing, A. M. (1950).** [*Computing Machinery and Intelligence*](https://redirect.cs.umbc.edu/courses/471/papers/turing.pdf). *Mind* 59(236), 433–460.
   - *(i) From brains to models.* The lower-left cell — [acting humanly](https://en.wikipedia.org/wiki/Turing_test) — is the imitation game. Turing replaces “can machines think?” with a verbal indistinguishability test. Large language models now pass ever more demanding versions of that game.
   - *(ii) What AI reveals about being human.* Turing sets “the mystery of [consciousness](https://plato.stanford.edu/entries/consciousness/)” aside. The paper does not: passing the game is unsurprising once the game *is* text. Theme (ii) begins where the judge’s transcript ends.

3. **Binz, M., and Schulz, E. (2023).** [*Using cognitive psychology to understand GPT-3*](https://www.pnas.org/doi/10.1073/pnas.2218523120). *PNAS* 120(6), e2218523120.
   - *(i) From brains to models.* The upper-left cell at scale: does the model reproduce human [errors](https://en.wikipedia.org/wiki/Framing_effect_(psychology))? Sometimes yes, sometimes no; vignettes are brittle and possibly memorised. Notebook 04 is the controlled demonstration; this map only places that result.
   - *(ii) What AI reveals about being human.* Matching a bias is not sharing a developmental history. The empty “thinking rationally” companion in the exercises is the complementary warning: a scratchpad is not a proof system.

4. **Ouyang, L., et al. (2022).** [*Training language models to follow instructions with human feedback*](https://arxiv.org/abs/2203.02155). *NeurIPS* 35. arXiv:2203.02155.
   - *(i) From brains to models.* The lower-right cell — acting rationally — arrives inverted: capability first, then a preference-tuned interface. Goals are imposed by the prompt and the rater, not by a utility that is the agent’s.
   - *(ii) What AI reveals about being human.* A preferred token is not a preferred observation in a world under [viability](https://en.wikipedia.org/wiki/Allostasis) constraints. Acting rationally in the textbook sense needs a utility that belongs to the agent. RLHF does not supply one.

5. **Bommasani, R., et al. (2021).** [*On the Opportunities and Risks of Foundation Models*](https://arxiv.org/abs/2108.07258). arXiv:2108.07258.
   - *(i) From brains to models.* Why one system can sit in every cell at once: a [foundation model](https://en.wikipedia.org/wiki/Foundation_model) is a single adapted predictor scored as imitation, as psychology, as a scratchpad, and as an agent.
   - *(ii) What AI reveals about being human.* Occupying every cell is not occupying a life. The report’s risks are social. The paper’s remainder is constitutive: embodiment, need, experience.

6. **Gabriel, I. (2020).** [*Artificial intelligence, values, and alignment*](https://arxiv.org/pdf/2001.09768). *Minds and Machines* 30, 411–437. arXiv:2001.09768.
   - *(i) From brains to models.* The acting-rationally cell, made normative: align with instructions, intentions, revealed preferences, ideal preferences, interests, or values — these are different targets. A principle-based approach is not the discovery of a true moral theory ([ethics of AI](https://plato.stanford.edu/entries/ethics-ai/)).
   - *(ii) What AI reveals about being human.* Fair principles for alignment, in a plural moral world, are a human political problem. The model does not have that problem. We do. That is theme (ii) for the lower-right cell.


### Exercises

1. **One system, four scores.** The map is four *questions*, not four machines. For each cell, print `lookup` on a word from that cell’s verdict (`errors`, `proof`, `imitation`, `goals`). Confirm you get one cell each time. Then write one sentence — not a new experiment — stating how a single clinical note-completer can be scored in all four cells at once.

<!-- solution-box: 04b_1_one_system_four_scores.py -->
<details>
<summary><strong>Show solution</strong> — One system, four scores</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/04b_1_one_system_four_scores.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.four_views import format_quadrant, lookup
for query in ("errors", "proof", "imitation", "goals"):
    hits = lookup(query)
    print(f"query {query!r}  →  {len(hits)} cell(s)")
    for cell in hits:
        print(" ", cell.title, "—", cell.verdict)
    print()
print(
    "A note-completer can copy a documented bias from discharge text "
    "(thinking humanly), emit a fluent but unproved plan (thinking "
    "rationally), pass a short Turing-style screen (acting humanly), and "
    "still have its objective added after pretraining (acting rationally). "
    "One architecture; four scores. Notebook 04 is the upper-left toy. "
    "The other three cells are verdicts, not extra nets."
)
```

</details>
<!-- /solution-box -->


2. **The empty companion.** Which cell has no companion notebook? Print it. Why did the paper refuse a fourth toy there, and what would a misleading toy have looked like (a longer prefix, a chain-of-thought string)?

<!-- solution-box: 04b_2_the_empty_companion.py -->
<details>
<summary><strong>Show solution</strong> — The empty companion</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/04b_2_the_empty_companion.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.four_views import QUADRANTS, format_quadrant
for cell in QUADRANTS:
    flag = cell.companion or "(none)"
    print(f"{cell.title:<22}  companion {flag}")
print()
empty = [c for c in QUADRANTS if not c.companion]
print(format_quadrant(empty[0]))
print(
    "\nThinking rationally has no extra toy because a scratchpad is still "
    "next-token prediction with a longer prefix. A notebook that printed "
    "a 'therefore' and called it a proof would be the misleading toy: it "
    "would look like System 2 and implement the same objective as "
    "notebook 01. The empty companion is the claim."
)
```

</details>
<!-- /solution-box -->


3. **Lookup is not a proof.** `lookup` is a substring match on the catalogue text. Try `FOCUS = "human"` and `FOCUS = "not a word in the catalogue"`. How many cells fire, and what does that tell you about treating the map as an experiment?

<!-- solution-box: 04b_3_lookup_is_not_a_proof.py -->
<details>
<summary><strong>Show solution</strong> — Lookup is not a proof</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/04b_3_lookup_is_not_a_proof.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.four_views import format_quadrant, lookup
for q in ("human", "not a word in the catalogue", "04", "02b"):
    hits = lookup(q)
    print(f"{q!r:40}  {len(hits)} hit(s):", ", ".join(c.title for c in hits) or "—")
print()
print(
    "The word 'human' sits in two cells by construction (the left column). "
    "A nonsense string sits in none. This notebook is an index, like "
    "notebook 00. It does not train, and a lookup hit is not evidence "
    "that a foundation model 'is' that cell. Notebook 04 is where a "
    "designed corpus is put on the table."
)
```

</details>
<!-- /solution-box -->
